In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [3]:
train = pd.read_csv('train_fe.csv')
test = pd.read_csv('test_fe.csv')

In [4]:
X = train.drop(columns=['id', 'PitNextLap'])
y = train['PitNextLap']
X_test = test.drop(columns=['id'])

cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [5]:
import optuna


def objective(trial):
    param = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': 10,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**param)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        preds = model.predict_proba(X_va)[:, 1]
        cv_scores.append(roc_auc_score(y_va, preds))

    return np.mean(cv_scores)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=25) 

print("\nBest hyperparameters found:")
for key, value in study.best_params.items():
    print(f"'{key}': {value},")
print(f"\nBest average ROC AUC (3-fold): {study.best_value:.5f}")

[I 2026-11-04 21:14:54,776] A new study created in memory with name: no-name-1b9989a3-e071-4918-b25c-ceb6457e0126
[I 2026-11-04 21:15:00,370] Trial 0 finished with value: 0.9256037437068847 and parameters: {'learning_rate': 0.025016379908805713, 'num_leaves': 102, 'max_depth': 11, 'min_child_samples': 158, 'subsample': 0.994457089394919, 'colsample_bytree': 0.9751114233191867, 'reg_alpha': 5.5608789567567645e-06, 'reg_lambda': 0.006346776156940034}. Best is trial 0 with value: 0.9256037437068847.
[I 2026-11-04 21:15:02,719] Trial 1 finished with value: 0.9163609615883276 and parameters: {'learning_rate': 0.04767529407542923, 'num_leaves': 20, 'max_depth': 11, 'min_child_samples': 90, 'subsample': 0.7406279146216046, 'colsample_bytree': 0.9094215490129585, 'reg_alpha': 0.017502943260638674, 'reg_lambda': 0.001955584874595382}. Best is trial 0 with value: 0.9256037437068847.
[I 2026-11-04 21:15:06,679] Trial 2 finished with value: 0.9282874414907815 and parameters: {'learning_rate': 0.05


Best hyperparameters found:
'learning_rate': 0.012551783888533855,
'num_leaves': 150,
'max_depth': 9,
'min_child_samples': 91,
'subsample': 0.9358033835109033,
'colsample_bytree': 0.7966369013834703,
'reg_alpha': 1.8768797222072426e-06,
'reg_lambda': 0.3907966181779994,

Best average ROC AUC (3-fold): 0.93136


In [6]:
best_params = {
    'n_estimators': 1000, 
    'learning_rate': 0.012551783888533855,
    'num_leaves': 150,
    'max_depth': 9,
    'min_child_samples': 91,
    'subsample': 0.9358033835109033,
    'colsample_bytree': 0.7966369013834703,
    'reg_alpha': 1.8768797222072426e-06,
    'reg_lambda': 0.3907966181779994,
    'scale_pos_weight': 10,
    'random_state': 42,
    'n_jobs': -1
}


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**best_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        categorical_feature=cat_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    val_probs = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_probs
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    print(f"Fold {fold+1} ROC AUC: {roc_auc_score(y_va, val_probs):.5f}")

total_auc = roc_auc_score(y, oof_preds)
print(f"\nCałkowity OOF ROC AUC: {total_auc:.5f}")

submission = pd.read_csv('sample_submission.csv')
submission['PitNextLap'] = test_preds
submission.to_csv('lgbm_optuna_submission.csv', index=False)

Fold 1 ROC AUC: 0.93235
Fold 2 ROC AUC: 0.93091
Fold 3 ROC AUC: 0.93229
Fold 4 ROC AUC: 0.93061
Fold 5 ROC AUC: 0.93210

Całkowity OOF ROC AUC: 0.93164
